In [1]:
!pip install pyreadstat

import pandas as pd
import pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.5 MB/s eta 0:00:00


In [2]:
ch = pd.read_csv("../../data/raw/ch_combined.csv")
print(f"Rows: {ch.shape[0]:,} | Columns: {ch.shape[1]}")

/tmp/ipykernel_6539/121900292.py:1: DtypeWarning: Columns (34,35,36,37,38,39,51,53,58,60,63,68,73,78,214,215,216,217,220,221,222,223,224,225,226,227,233,234,235,236,237,241,242,244,245,246,247,248,250,251,252,253,254,257,258,259,260,261,262,263,264,265,272,273,276,277,286,287,288,289,292,293,294,295,296,297,298,299,301,302,303,304,305,306,307,308,309,311,318,319,321,322,323,324,325,327,329,330,331,332,333,334,335,336,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353) have mixed types. Specify dtype option on import or set low_memory=False.
  ch = pd.read_csv("ch_combined.csv")


Rows: 45,231 | Columns: 423


In [3]:
cols_to_keep = [
    'HH1', 'HH2',          # Merge keys
    'CAGE',                 # Child age in months
    'HL4',                  # Child sex
    'BR2',                  # Birth weight
    'BR3',                  # Breastfeeding
    'HAZ', 'WAZ', 'WHZ',   # Z scores
    'HAZ2', 'WAZ2', 'WHZ2',# Stunted/Underweight/Wasted
    'melevel',              # Mother education
    'HH6',                  # Urban/Rural
    'HH7',                  # District
    'windex5',              # Wealth index
    'cdisability',          # Disability
    'province'              # Province
]

ch_clean = ch[cols_to_keep].copy()

ch_clean.columns = [
    'cluster_id', 'household_id',
    'child_age_months', 'child_sex',
    'birth_weight', 'breastfeeding',
    'haz_score', 'waz_score', 'whz_score',
    'stunted', 'underweight', 'wasted',
    'mother_education', 'urban_rural',
    'district', 'wealth_index',
    'disability', 'province'
]

print(f"Rows: {ch_clean.shape[0]:,} | Columns: {ch_clean.shape[1]}")
print(ch_clean.isnull().sum())

Rows: 45,231 | Columns: 18
cluster_id              0
household_id            0
child_age_months      323
child_sex               1
birth_weight        16735
breastfeeding       17955
haz_score             323
waz_score             323
whz_score             323
stunted               323
underweight           323
wasted                324
mother_education        1
urban_rural             1
district                1
wealth_index            1
disability          15773
province                1
dtype: int64


In [4]:
# Fix target variable — convert Z scores to 1/0
ch_clean['stunted']     = (ch_clean['stunted'] < -2).astype(int)
ch_clean['wasted']      = (ch_clean['wasted'] < -2).astype(int)
ch_clean['underweight'] = (ch_clean['underweight'] < -2).astype(int)

# Drop rows where age or z score is missing
ch_clean = ch_clean.dropna(subset=['child_age_months', 'haz_score'])
print(f"After dropping missing rows: {ch_clean.shape[0]:,}")

# birth_weight — too many missing, just flag it
ch_clean['birth_weight_known'] = ch_clean['birth_weight'].notna().astype(int)
ch_clean = ch_clean.drop(columns=['birth_weight'])

# breastfeeding — fill with mode
ch_clean['breastfeeding'] = ch_clean['breastfeeding'].fillna(
    ch_clean['breastfeeding'].mode()[0])

# disability — fill with 0
ch_clean['disability'] = ch_clean['disability'].fillna(0)

# mother_education — fill with mode
ch_clean['mother_education'] = ch_clean['mother_education'].fillna(
    ch_clean['mother_education'].mode()[0])

# Encode child_sex: 1=male→0, 2=female→1
ch_clean['child_sex'] = ch_clean['child_sex'].map({1.0: 0, 2.0: 1})

# Encode urban_rural: 1=urban→0, 2=rural→1
ch_clean['urban_rural'] = ch_clean['urban_rural'].map({1.0: 0, 2.0: 1})

# Encode province
ch_clean['province'] = ch_clean['province'].map({
    'Balochistan': 0,
    'KPK': 1,
    'Sindh': 2
})

After dropping missing rows: 44,908


In [5]:
print(f"Final shape: {ch_clean.shape[0]:,} rows | {ch_clean.shape[1]} columns")
print(f"\nStunting rate: {ch_clean['stunted'].mean()*100:.1f}%")
print(f"\nMissing values:")
print(ch_clean.isnull().sum())
print(f"\nColumns: {ch_clean.columns.tolist()}")

# Save
ch_clean.to_csv("../../data/processed/ch_cleaned.csv", index=False)
print("\nSaved!")

Final shape: 44,908 rows | 18 columns

Stunting rate: 40.9%

Missing values:
cluster_id            0
household_id          0
child_age_months      0
child_sex             0
breastfeeding         0
haz_score             0
waz_score             0
whz_score             0
stunted               0
underweight           0
wasted                0
mother_education      0
urban_rural           0
district              0
wealth_index          0
disability            0
province              0
birth_weight_known    0
dtype: int64

Columns: ['cluster_id', 'household_id', 'child_age_months', 'child_sex', 'breastfeeding', 'haz_score', 'waz_score', 'whz_score', 'stunted', 'underweight', 'wasted', 'mother_education', 'urban_rural', 'district', 'wealth_index', 'disability', 'province', 'birth_weight_known']

Saved!
